# **EUROPEAN MONARCHIES NETWORK**
Our final project consists of a study of the european royal families. We study the members of each family, how they are related to each other, and how they are related to other members of other families. The main pieces of interesting information would be discovering family relationships between different royal houses or countries, because different royal houses from different countries are related to each other due to historical reasons, for example marriages to form alliances or increase power.

It would be interesting to be able to identify common ancestors of the actual Kings and Queens of Europe, like for example Queen Victoria of the United Kingdom.

To do this, the objective is to create a network with the following characteristics:

Nodes:


*   All Kings, Queens, Spouses, and brothers (to expand the network, we can explore the cousins of the monarchs too)
*   Node size based on importance (priority in the line of succession). Being monarchs the nodes with the biggest size.
*   Circular nodes with the images of the character which that node represents.
*   Border colour of the node based on the country or the royal household from the character that the node represents.

Edges:


*   They represent the relationship between two characters or nodes.
*   We can have different colours for different types of relationships (Direct succession/predecession, husband or wife, brotherhood, sons which never accessed the throne)
*   Different types of relationships can be also modeled using width of edges.

**Corrected version:** retains the original 18 starting monarchs and original 1450 death cutoff. The `country` field is never stored on graph nodes; overlapping blood/throne relationships are combined without changing the node set.

**Versione QID.** Nodi, archi, parents e visite usano identificativi Wikidata.
`name`/`label` e `wikipedia_title` sono attributi di visualizzazione. Output in `dati_qid/`.
La raccolta live conserva seed, cutoff e filtri del notebook fornito, ma non carica i vecchi grafi con chiavi nominali.
Eseguire in ordine fino alla cella che stampa “Output QID salvati”. Le sezioni testuali successive sono facoltative e richiedono il corpus.
La vecchia variabile TIMESTAMP_SNAPSHOT non congelava le entità scaricate: questa versione non dichiara uno snapshot storico.
Le regole di classificazione monarch/religione e le interpretazioni degli attributi restano quelle del file fornito, non validate da questa modifica.


**Aggiornamento 19 novembre:** raccolta QID limitata a persone con pagina enwiki; le altre non vengono esplorate. Seconda cella di codice commentata, stato inizializzato negli import. La cache SQLite resta utilizzabile, ma ogni entità viene filtrata di nuovo. Seed e cutoff correnti conservati.


# ***Important!!! ***

If some issues occur in running the code, figures could be seen [here](https://nbviewer.org/github/Davide011/Social_Graph/blob/main/COLAB_FINAL_PROJECT.ipynb) .



In [ ]:
import string
import os
import requests
import re
import pickle
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import ipywidgets as widgets
import nltk
from nltk.corpus import PlaintextCorpusReader, stopwords
from nltk import word_tokenize
from nltk.text import TextCollection
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from wordcloud import WordCloud
from collections import defaultdict
from IPython.display import display
from pathlib import Path
from urllib.parse import unquote
from html import unescape

# Output QID separati dai grafi precedenti basati sui nomi.
RUN_DIR = Path.cwd().resolve()
if (RUN_DIR / 'finaleee').is_dir():
    PROJECT_DIR = RUN_DIR / 'finaleee'
else:
    PROJECT_DIR = RUN_DIR
DATA_DIR = PROJECT_DIR / 'dati_qid'
FINAL_DATA_DIR = DATA_DIR
TEXT_DIR = PROJECT_DIR / 'fonti' / 'monarch_text'
NLTK_DATA_DIR = DATA_DIR / 'nltk_data'
GRAPH_FILE = DATA_DIR / 'graph_relations_qid.pkl'
LABMT_FILE = PROJECT_DIR / 'fonti' / 'Data_Set_S1.txt'
DATA_DIR.mkdir(parents=True, exist_ok=True)
# Raccolta live: non è uno snapshot storico al 2026-01-01.
# Le risposte usate vengono archiviate a fine raccolta per tracciabilità.

def atomic_pickle_dump(value, path):
    """Write a pickle through a temporary file so an interrupted save keeps the old cache."""
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    try:
        with temporary.open("wb") as file:
            pickle.dump(value, file)
        os.replace(temporary, path)
    except BaseException:
        if temporary.exists():
            temporary.unlink()
        raise

# Il generatore del grafo non richiede download NLTK.
nltk.data.path.insert(0, str(NLTK_DATA_DIR))

# Stato iniziale necessario anche con la seconda cella di codice commentata.
FORCE_REBUILD_GRAPH = False
G = None
LOAD_SAVED_GRAPH = False


Load a full graph from a python .pkl file (if you run this, skip all the network construction part)

In [ ]:
# CARICAMENTO GRAFO DISABILITATO: riattivare solo per un grafo con gli stessi criteri.
# FORCE_REBUILD_GRAPH = False
# G = None
# LOAD_SAVED_GRAPH = False
# if GRAPH_FILE.is_file() and not FORCE_REBUILD_GRAPH:
#     with GRAPH_FILE.open('rb') as file:
#         G = pickle.load(file)
#     if (not isinstance(G, nx.Graph) or G.is_directed() or G.is_multigraph()
#             or not G or G.graph.get('identity_schema') != 'wikidata_qid_v1'):
#         raise ValueError('Cache non compatibile: serve un grafo QID v1 non vuoto.')
#     for qid, attrs in G.nodes(data=True):
#         if not isinstance(qid, str) or not re.fullmatch(r'Q[1-9][0-9]*', qid):
#             raise ValueError(f'Identificativo non QID nella cache: {qid!r}')
#         if attrs.get('wikidata_qid') != qid:
#             raise ValueError(f'Attributo wikidata_qid incoerente: {qid}')
#         if not isinstance(attrs.get('parents'), list) or any(
#             not isinstance(p, str) or not re.fullmatch(r'Q[1-9][0-9]*', p)
#             for p in attrs['parents']
#         ):
#             raise ValueError(f'Genitori non QID nella cache: {qid}')
#     LOAD_SAVED_GRAPH = True
#     print(f'Cache QID caricata: {len(G)} nodi, {G.number_of_edges()} archi')
# else:
#     print('Nuova raccolta Wikidata in QID; nessuna cache basata sui nomi viene importata.')


# NETWORK CONSTRUCTION

We will start from a list of the 18 last/latest monarchs in different countries.

In [ ]:

# Given data
data = {
   # 'country': ['Belgium', 'Bulgaria', 'Denmark', 'Greece', 'Italy', 'Luxembourg', 'Montenegro',
    #            'Netherlands', 'Norway', 'Portugal', 'Romania', 'Russia', 'Serbia', 'Spain', 'Sweden', 'UK', 'Liechtenstein', 'Monaco'],

    'country': [ 'UK'],

   # 'monarch_name': ['Philippe of Belgium', 'Simeon Saxe-Coburg-Gotha', 'Margrethe II of Denmark', 'Constantine II of Greece',
    #                 'Umberto II of Italy', 'Henri, Grand Duke of Luxembourg', 'Nicholas I of Montenegro', 'Willem-Alexander of the Netherlands',
     #                'Harald V of Norway', 'King Manuel II', 'Michael I of Romania', 'Nicholas II of Russia', 'Peter I of Serbia',
      #               'Felipe VI', 'Carl XVI Gustaf', 'Charles III', 'Hans-Adam II, Prince of Liechtenstein', 'Albert II, Prince of Monaco'],

      'monarch_name': [ 'Charles III'],

   # 'time': ['2013 - Present', 'Ended in 1946', '1972 - Present', 'Ended in 1973', 'Effectively ended in 1946',
    #         '2000 - Present', 'Ended in 1918', '2013 - Present', '1991 - Present', 'Ended in 1910',
     #        '1927-1930, 1930-1937', 'Ended in 1917', 'Ended in 1921', '2014 - Present', '1973 - Present', '1952 - Present', '2004 - Present', '2005 - Present']

     'timme':[ ' 1952 - Present']
}



# Creating a Pandas DataFrame
df = pd.DataFrame(data)
# Adding a new column with monarch names spaced by underscores
df['monarch_name_spaced'] = df['monarch_name'].str.replace(' ', '_')

df.head()

Many Wikipedia articles have multiple links that redirect to them. If we want to get the correct data for nodes, we have to get the original article that a name/string links to. We can do this via API calls, but that is very costly for graphs with a lot of node, so Instead we've built a link-article dictionary that stores all the redirects we've come across. It can be loaded from a pickle (.pkl) file.

In [ ]:
# Non occorrono redirect fra nomi per costruire gli archi QID.
# Le cache API e le diagnostiche sono inizializzate nella cella seguente.


Here we define auxiliary functions to deal with wikipedia information: getting wikipedia articles from their links, links from text, redirects from links...

In [ ]:
import time
import requests
import sqlite3
import json
from contextlib import closing

# Ogni risposta riuscita viene conservata; rilanciare ricostruisce usando la cache.
ENTITY_CACHE_FILE = DATA_DIR / 'wikidata_entities_cache_v1.sqlite'
ENTITY_CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
with closing(sqlite3.connect(ENTITY_CACHE_FILE)) as db:
    db.execute('CREATE TABLE IF NOT EXISTS entities (qid TEXT PRIMARY KEY, payload TEXT NOT NULL)')
    db.commit()
HTTP_SESSION = requests.Session()
HTTP_STATS = {'requests': 0, 'disk_hits': 0}
BATCH_SIZE = 50
collection_started = time.monotonic()

def _disk_restore(qids):
    missing = list(dict.fromkeys(q for q in qids if q not in entity_cache))
    if not missing:
        return
    with closing(sqlite3.connect(ENTITY_CACHE_FILE)) as db:
        for offset in range(0, len(missing), 50):
            chunk = missing[offset:offset+50]
            sql = 'SELECT qid, payload FROM entities WHERE qid IN (' + ','.join('?' for _ in chunk) + ')'
            for qid, payload in db.execute(sql, chunk):
                entity = json.loads(payload)
                entity_cache[qid] = entity
                if entity:
                    qid_aliases[qid] = entity.get('id', qid)
                    entity_cache[entity.get('id', qid)] = entity
                HTTP_STATS['disk_hits'] += 1

def _disk_store(qids):
    records = [(q, json.dumps(entity_cache[q],ensure_ascii=False)) for q in dict.fromkeys(qids) if q in entity_cache]
    if records:
        with closing(sqlite3.connect(ENTITY_CACHE_FILE)) as db:
            db.executemany('INSERT OR REPLACE INTO entities VALUES (?,?)', records)
            db.commit()

def prefetch_entities(qids):
    qids = list(dict.fromkeys(q for q in qids if is_qid(q)))
    _disk_restore(qids)
    missing = [q for q in qids if q not in entity_cache]
    for offset in range(0, len(missing), BATCH_SIZE):
        chunk = missing[offset:offset+BATCH_SIZE]
        payload = _wd_get({'action':'wbgetentities','ids':'|'.join(chunk),
                           'props':'labels|claims|sitelinks','redirects':'yes',
                           'languages':'en','format':'json'})
        stored = []
        entities = payload.get('entities', {})
        if not isinstance(entities, dict):
            raise RuntimeError('Risposta batch priva di entities valido')
        for key, entity in entities.items():
            if not is_qid(key) or not isinstance(entity, dict):
                continue
            if 'missing' in entity:
                entity_cache[key] = None
                stored.append(key)
            elif is_qid(entity.get('id')):
                actual = entity['id']
                entity_cache[key] = entity
                entity_cache[actual] = entity
                qid_aliases[key] = actual
                stored.extend([key,actual])
        redirects = payload.get('redirects', [])
        if isinstance(redirects, list):
            for item in redirects:
                if not isinstance(item, dict):
                    continue
                old, new = item.get('from'), item.get('to')
                if is_qid(old) and is_qid(new) and new in entity_cache:
                    entity_cache[old] = entity_cache[new]
                    qid_aliases[old] = new
                    stored.append(old)
        _disk_store(stored)
        # QID non risolti esplicitamente: fallback individuale in get_wikidata_entity.

def prefetch_metadata(qids):
    refs = []
    for qid in qids:
        entity = entity_cache.get(qid)
        if not entity:
            continue
        for prop in ('P19','P27','P140','P53','P21'):
            for value, _ in _claim_values(entity.get('claims', {}),prop):
                if isinstance(value, dict) and is_qid(value.get('id')):
                    refs.append(value['id'])
    prefetch_entities(refs)
    countries = []
    for qid in refs:
        entity = entity_cache.get(qid)
        if entity:
            for value, _ in _claim_values(entity.get('claims', {}),'P17'):
                if isinstance(value, dict) and is_qid(value.get('id')):
                    countries.append(value['id'])
    prefetch_entities(countries)


WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"
WD_HEADERS = {
    "User-Agent": "CrownGraph/2.0 (https://github.com/Davide011/CrownGraph; wikidata backend)",
    "Accept": "application/json",
}

# Cache: wikipedia_title -> QID  (None = non trovato o non persona)
qid_cache: dict = {}
# Cache: QID -> entity dict
entity_cache: dict = {}
# Cache: QID -> wikipedia EN title
title_cache: dict = {}
qid_aliases = {}
exclusion_reasons = {}
api_errors = {}
seed_resolution = []
discarded_edges = []


def is_qid(value):
    return isinstance(value, str) and re.fullmatch(r'Q[1-9][0-9]*', value) is not None


def canonical_qid(qid):
    seen = set()
    while qid in qid_aliases and qid_aliases[qid] != qid:
        if qid in seen:
            raise ValueError(f'Ciclo nei redirect QID: {qid}')
        seen.add(qid)
        qid = qid_aliases[qid]
    return qid



def _wd_get(params: dict) -> dict:
    """Sessione persistente, retry finiti e rispetto dei segnali di sovraccarico."""
    params = dict(params, maxlag=5)
    for attempt in range(4):
        HTTP_STATS['requests'] += 1
        response = HTTP_SESSION.get(WIKIDATA_API_URL, params=params,
                                    headers=WD_HEADERS, timeout=30)
        if response.status_code in (429, 500, 502, 503, 504):
            if attempt == 3:
                response.raise_for_status()
            wait = response.headers.get('Retry-After','')
            time.sleep(float(wait) if wait.isdigit() else 2 ** (attempt+1))
            continue
        response.raise_for_status()
        payload = response.json()
        if not isinstance(payload, dict):
            raise RuntimeError('Risposta JSON inattesa')
        error = payload.get('error')
        if error:
            if error.get('code') in ('maxlag','ratelimited') and attempt < 3:
                time.sleep(2 ** (attempt+1))
                continue
            raise RuntimeError(f'Errore API Wikidata: {error!r}')
        return payload
    raise RuntimeError('Tentativi API esauriti')


def _resolve_qid_by_title(title: str) -> str | None:
    """Chiama wbgetentities con sites=enwiki per un dato titolo.
    Ritorna il QID se trovato, None altrimenti. Non usa la cache."""
    try:
        payload = _wd_get({
            "action": "wbgetentities",
            "sites": "enwiki",
            "titles": title,
            "props": "ids",
            "format": "json",
            "redirects": "yes",
        })
    except Exception as exc:
        api_errors[title] = f"{type(exc).__name__}: {exc}"
        raise

    for entity_id, entity in payload.get("entities", {}).items():
        # Wikidata restituisce "-1" se l'articolo non è mappato
        if "missing" in entity:
            continue
        qid = entity.get("id", entity_id)
        if isinstance(qid, str) and qid.startswith("Q"):
            return qid
    return None


def _normalize_title(title: str) -> list[str]:
    """Genera varianti del titolo Wikipedia per aumentare la probabilità di match in Wikidata.
    Wikidata usa spesso forme più brevi (es. 'Albert II of Monaco' invece di
    'Albert II, Prince of Monaco').
    """
    variants = [title]
    # Rimuove il rank dopo la virgola: "Albert II, Prince of Monaco" → "Albert II of Monaco"
    import re as _re
    m = _re.match(r'^(.+?),\s*(Prince|Princess|Duke|Duchess|Count|Countess|King|Queen|'
                  r'Grand Duke|Grand Duchess|Archduke|Archduchess|Emperor|Empress|'
                  r'Elector|Electress|Margrave|Landgrave|Earl|Baron|Lord|Lady)\s+of\s+(.+)$',
                  title)
    if m:
        # "Albert II of Monaco"
        variants.append(f"{m.group(1)} of {m.group(3)}")
    # Aggiunge la versione con _ sostituiti da spazi (già lo è, ma per sicurezza)
    variants.append(title.replace("_", " "))
    # Rimuove eventuale disambiguazione tra parentesi: "X (1234–5678)" → "X"
    m2 = _re.match(r'^(.+?)\s*\(\d', title)
    if m2:
        variants.append(m2.group(1).strip())
    # Deduplica mantenendo ordine
    seen, out = set(), []
    for v in variants:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out


def get_wikidata_qid(wikipedia_title: str) -> str | None:
    """Dato un titolo Wikipedia EN, restituisce il QID Wikidata (es. 'Q3910').
    Accetta anche QID già noti. Per i seed usa il titolo esatto e i redirect API.
    Non elimina disambiguazioni; usa qid_cache per evitare chiamate ridondanti."""
    if not wikipedia_title:
        return None
    key = wikipedia_title.strip()
    if key in qid_cache:
        return qid_cache[key]

    if is_qid(key):
        return canonical_qid(key)
    qid = _resolve_qid_by_title(key.replace('_', ' '))
    qid_cache[key] = qid
    return qid


def get_wikidata_entity(qid: str) -> dict | None:
    """Scarica (con cache) l'entity JSON completa da Wikidata per un dato QID."""
    if not qid:
        return None
    _disk_restore([qid])
    if qid in entity_cache:
        return entity_cache[qid]

    try:
        payload = _wd_get({
            "action": "wbgetentities",
            "ids": qid,
            "props": "labels|claims|sitelinks",
            "redirects": "yes",
            "languages": "en",
            "format": "json",
        })
    except Exception as exc:
        api_errors[qid] = f"{type(exc).__name__}: {exc}"
        raise  # Non trasformare un guasto di rete in una persona esclusa.

    entities = payload.get('entities', {})
    entity = entities.get(qid)
    valid = [e for e in entities.values()
             if isinstance(e, dict) and 'missing' not in e and is_qid(e.get('id'))]
    if (entity is None or 'missing' in entity) and len(valid) == 1:
        entity = valid[0]
    if not entity or 'missing' in entity:
        entity_cache[qid] = None
        _disk_store([qid])
        return None
    resolved = entity.get('id', qid)
    qid_aliases[qid] = resolved
    entity_cache[qid] = entity
    entity_cache[resolved] = entity
    _disk_store([qid,resolved])
    return entity


def get_entity_wikipedia_title(qid: str) -> str | None:
    """Dato un QID, restituisce il titolo Wikipedia EN (solo metadato; non identifica nodi o archi)."""
    if not qid:
        return None
    if qid in title_cache:
        return title_cache[qid]

    entity = get_wikidata_entity(qid)
    if entity is None:
        title_cache[qid] = None
        return None
    title = entity.get("sitelinks", {}).get("enwiki", {}).get("title")
    title_cache[qid] = title
    return title


def _claim_values(claims: dict, prop: str) -> list:
    """Estrae i datavalue di una proprietà dai claims Wikidata (solo mainsnak value)."""
    results = []
    for claim in claims.get(prop, []):
        snak = claim.get("mainsnak", {})
        if snak.get("snaktype") == "value":
            dv = snak.get("datavalue", {})
            results.append((dv.get("value"), claim))
    return results


def _year_from_time(time_str: str) -> str | None:
    """Estrae l'anno (4 cifre) da una stringa ISO-like di Wikidata (es. '+1830-01-01T00:00:00Z')."""
    m = re.search(r"\b(\d{4})\b", time_str)
    return m.group(1) if m else None

# Import facoltativo di entità salvate da una precedente esecuzione interrotta.
archive = DATA_DIR / 'wikidata_entities_used.pkl'
if archive.is_file():
    with archive.open('rb') as stream:
        saved_entities = pickle.load(stream)
    if not isinstance(saved_entities, dict):
        raise ValueError('Archivio entità non valido')
    _disk_restore(list(saved_entities))
    for key, entity in saved_entities.items():
        if is_qid(key) and key not in entity_cache and isinstance(entity, dict) and is_qid(entity.get('id')):
            entity_cache[key] = entity
            qid_aliases[key] = entity['id']
    _disk_store(entity_cache.keys())
    del saved_entities


Function to build a node from a given article name/link. Returns a node item as well as its neighbors in the network. The built nodes will have some attributes from Wikidata:
- monarch: wether the person is reigning or has reigned over something in the past.
- house: the royal family the person belongs to
- reign_start: the start of the person's reign, if it has one
- birth/death: the person's date of birth and death (if present)
- parents: the person's parents
- sex / gender: sex or gender of the individual (P21 from Wikidata, e.g. 'male', 'female')
- wikidata_qid: the Wikidata QID of the entity

Appart from this, the function will return the person's parents, children and predecessor (if the person has one)

In [ ]:
def normalize_religion(raw_name: str, qid: str = None) -> str | None:
    """Normalizza denominazioni ed etichette equivalenti di religione in una categoria canonica condivisa."""
    if not raw_name and not qid:
        return None

    # Mappatura diretta per QID noti di Wikidata
    QID_RELIGION_MAP = {
        # Cattolicesimo
        "Q9592": "Catholicism",    # Catholic Church / Chiesa cattolica
        "Q1841": "Catholicism",    # Catholicism / cattolicesimo
        "Q134142": "Catholicism",  # Latin Church / Chiesa latina
        "Q188619": "Catholicism",  # Roman Catholic
        # Anglicanesimo
        "Q6423963": "Anglicanism", # Anglicanism / anglicanesimo
        "Q82708": "Anglicanism",   # Church of England / Chiesa anglicana
        "Q228828": "Anglicanism",  # Scottish Episcopal Church
        # Luteranesimo
        "Q75809": "Lutheranism",   # Lutheranism / luteranesimo
        "Q855585": "Lutheranism",  # Church of Denmark / Chiesa di Danimarca
        "Q1074944": "Lutheranism", # Church of Sweden / Chiesa di Svezia
        "Q489578": "Lutheranism",  # Church of Norway / Chiesa di Norvegia
        "Q170111": "Lutheranism",  # Evangelical Church in Germany
        # Calvinismo / Riformati / Presbiteriani
        "Q101849": "Calvinism",    # Reformed Christianity / Calvinism / calvinismo
        "Q178169": "Calvinism",    # Presbyterianism / presbiterianesimo
        "Q188449": "Calvinism",    # Church of Scotland / Chiesa di Scozia
        "Q575460": "Calvinism",    # Dutch Reformed Church
        # Ortodossia orientale
        "Q3333484": "Eastern Orthodoxy", # Eastern Orthodoxy / cristianesimo ortodosso
        "Q40866": "Eastern Orthodoxy",   # Eastern Orthodox Church
        "Q60995": "Eastern Orthodoxy",   # Russian Orthodox Church / Chiesa ortodossa russa
        "Q3472099": "Eastern Orthodoxy", # Greek Orthodox Church
        "Q200780": "Eastern Orthodoxy",  # Serbian Orthodox Church
        "Q181910": "Eastern Orthodoxy",  # Romanian Orthodox Church
        "Q191942": "Eastern Orthodoxy",  # Bulgarian Orthodox Church
        "Q207320": "Eastern Orthodoxy",  # Georgian Orthodox Church
        # Protestantesimo generico
        "Q23540": "Protestantism",  # Protestantism / protestantesimo
        # Altre religioni
        "Q5043": "Christianity",    # Christianity / cristianesimo
        "Q432": "Islam",            # Islam
        "Q483654": "Islam",         # Sunni Islam
        "Q9232": "Islam",           # Shia Islam
        "Q9268": "Judaism",         # Judaism / ebraismo
    }

    if qid and qid in QID_RELIGION_MAP:
        return QID_RELIGION_MAP[qid]

    if not raw_name:
        return qid

    text = str(raw_name).lower().strip()

    # Cattolicesimo
    if any(k in text for k in ("catholic", "cattolic", "latin church", "chiesa latina", "chiesa di roma")):
        return "Catholicism"

    # Ortodossia
    if any(k in text for k in ("orthodox", "ortodoss", "pravoslav")):
        return "Eastern Orthodoxy"

    # Anglicanesimo
    if any(k in text for k in ("anglican", "church of england", "england", "episcopal", "chiesa anglicana", "chiesa d'inghilterra")):
        return "Anglicanism"

    # Luteranesimo
    if any(k in text for k in (
        "lutheran", "luteran", "church of denmark", "church of sweden",
        "church of norway", "chiesa di danimarca", "chiesa di svezia", "chiesa di norvegia"
    )):
        return "Lutheranism"

    # Calvinismo / Presbiterianesimo / Chiese Riformate
    if any(k in text for k in (
        "calvinis", "reformed", "riformat", "presbyterian", "presbiterian",
        "church of scotland", "chiesa di scozia", "dutch reformed"
    )):
        return "Calvinism"

    # Protestantesimo generico
    if any(k in text for k in ("protestant", "protestantesimo", "evangelic", "evangelica")):
        return "Protestantism"

    # Ebraismo
    if any(k in text for k in ("judaism", "jewish", "ebraism", "ebreo", "ebraica","hebrew")):
        return "Judaism"

    # Islam
    if any(k in text for k in ("islam", "muslim", "sunni", "shia", "sunnit", "sciit")):
        return "Islam"

    # Cristianesimo generico
    if any(k in text for k in ("christian", "cristianes")):
        return "Christianity"

    return raw_name.strip()


def get_node(name: str):
    """Costruisce un nodo QID da un QID o da un titolo seed Wikipedia EN.
    Mantiene la stessa firma dell'originale:
      restituisce (node, predecessor, spouses, offspring, parents)
    dove node = (qid, attrs_dict); tutti i familiari sono QID.
    """
    if not name:
        return None, None, None, None, None

    qid = get_wikidata_qid(name)
    if not qid:
        return None, None, None, None, None

    entity = get_wikidata_entity(qid)
    if not entity:
        exclusion_reasons[qid] = 'entity_missing'
        return None, None, None, None, None
    # Perimetro del campione: solo persone con pagina Wikipedia inglese.
    # Un'entità senza enwiki non viene ammessa né espansa, anche se è in cache.
    if not entity.get('sitelinks', {}).get('enwiki', {}).get('title'):
        exclusion_reasons[entity.get('id', qid)] = 'no_enwiki_sitelink'
        return None, None, None, None, None
    qid = entity.get('id', qid)
    if not is_qid(qid):
        raise ValueError(f'Entity ID non valido: {qid!r}')

    claims = entity.get("claims", {})

    # ── Verifica che sia una persona (P31 = Q5) ──────────────────────────────
    p31_values = [v for v, _ in _claim_values(claims, "P31")]
    is_human = any(isinstance(v, dict) and v.get("id") == "Q5" for v in p31_values)
    if not is_human:
        exclusion_reasons[qid] = 'P31_not_Q5'
        return None, None, None, None, None

    # ── Nome canonico (etichetta EN) ─────────────────────────────────────────
    labels = entity.get("labels", {})
    canonical_name = labels.get("en", {}).get("value", qid)
    wikipedia_title = entity.get("sitelinks", {}).get("enwiki", {}).get("title")

    # ── Sesso o genere P21 ──────────────────────────────────────────────────
    GENDER_MAP = {
        "Q6581097": "male",
        "Q6581072": "female",
        "Q1052281": "female",
        "Q2449503": "male",
        "Q44148": "male",
        "Q43445": "female",
    }
    sex = None
    for v, _ in _claim_values(claims, "P21"):
        if isinstance(v, dict) and "id" in v:
            gid = v["id"]
            sex = GENDER_MAP.get(gid)
            if not sex:
                sex = get_entity_wikipedia_title(gid) or gid
            break

    # ── Data di nascita P569 ─────────────────────────────────────────────────
    birth = None
    for v, _ in _claim_values(claims, "P569"):
        if isinstance(v, dict):
            birth = _year_from_time(v.get("time", ""))
            if birth:
                break

    # ── Luogo e Stato di nascita (P19 -> P17) ────────────────────────────────
    birth_place = None
    birth_country = None
    for v, _ in _claim_values(claims, "P19"):
        if isinstance(v, dict) and "id" in v:
            p_qid = v["id"]
            p_entity = get_wikidata_entity(p_qid)
            if p_entity:
                birth_place = (
                    p_entity.get("labels", {}).get("en", {}).get("value")
                    or p_entity.get("sitelinks", {}).get("enwiki", {}).get("title")
                    or p_qid
                )
                # Ricaviamo lo Stato (P17: country) del luogo di nascita
                p_claims = p_entity.get("claims", {})
                for c_val, _ in _claim_values(p_claims, "P17"):
                    if isinstance(c_val, dict) and "id" in c_val:
                        c_qid = c_val["id"]
                        c_entity = get_wikidata_entity(c_qid)
                        if c_entity:
                            birth_country = (
                                c_entity.get("labels", {}).get("en", {}).get("value")
                                or c_entity.get("sitelinks", {}).get("enwiki", {}).get("title")
                                or c_qid
                            )
                        else:
                            birth_country = c_qid
                        if birth_country:
                            break
            else:
                birth_place = p_qid
            break

    # Fallback sullo stato di cittadinanza P27 se lo stato di nascita non è risolto
    if not birth_country and "P27" in claims:
        for c_val, _ in _claim_values(claims, "P27"):
            if isinstance(c_val, dict) and "id" in c_val:
                c_entity = get_wikidata_entity(c_val["id"])
                if c_entity:
                    birth_country = (
                        c_entity.get("labels", {}).get("en", {}).get("value")
                        or c_entity.get("sitelinks", {}).get("enwiki", {}).get("title")
                    )
                if birth_country:
                    break

    # ── Religione P140 (normalizzata) ────────────────────────────────────────
    religion = None
    for v, _ in _claim_values(claims, "P140"):
        if isinstance(v, dict) and "id" in v:
            r_qid = v["id"]
            r_entity = get_wikidata_entity(r_qid)
            raw_rel = None
            if r_entity:
                raw_rel = (
                    r_entity.get("labels", {}).get("en", {}).get("value")
                    or r_entity.get("sitelinks", {}).get("enwiki", {}).get("title")
                    or r_qid
                )
            else:
                raw_rel = r_qid
            religion = normalize_religion(raw_rel, r_qid)
            if religion:
                break

    # ── Data di morte P570 ───────────────────────────────────────────────────
    death = None
    for v, _ in _claim_values(claims, "P570"):
        if isinstance(v, dict):
            death = _year_from_time(v.get("time", ""))
            if death:
                break

    # ── Casa reale P53 (con fallback a P27 country of citizenship) ───────────
    house = None
    for v, _ in _claim_values(claims, "P53"):
        if isinstance(v, dict) and "id" in v:
            house = get_entity_wikipedia_title(v["id"]) or v["id"]
            break

    # ── Carica P39 → has_reign, reign_start ──────────────────────────────────
    # Proprieta monarchiche comuni in Wikidata
    MONARCHICAL_ROLES = {
        "Q116",    # monarch
        "Q1097",   # king
        "Q2461048", # queen
        "Q209182", # queen consort
        "Q46197",  # emperor
        "Q163062", # prince
        "Q179637", # grand duke
    }
    has_reign = False
    reign_start = None
    for v, claim in _claim_values(claims, "P39"):
        role_qid = v.get("id") if isinstance(v, dict) else None
        # Se la carica è esplicitamente monarchica o ha un start-time, la contiamo
        qualifiers = claim.get("qualifiers", {})
        start_quals = qualifiers.get("P580", [])
        for sq in start_quals:
            dv = sq.get("datavalue", {}).get("value", {})
            if isinstance(dv, dict) and "time" in dv:
                y = _year_from_time(dv["time"])
                if y:
                    has_reign = True
                    reign_start = y
                    break
        if has_reign:
            break
    # Se ha P39 ma senza start qualifier, è comunque un personaggio con carica
    if not has_reign and "P39" in claims:
        # Controlla se qualche carica è di tipo monarchico
        for v, _ in _claim_values(claims, "P39"):
            if isinstance(v, dict) and v.get("id") in MONARCHICAL_ROLES:
                has_reign = True
                break

    # Identità delle relazioni: conservare i QID delle proprietà, anche senza enwiki.
    def related_qids(*properties):
        return list(dict.fromkeys(
            v['id'] for prop in properties for v, _ in _claim_values(claims, prop)
            if isinstance(v, dict) and is_qid(v.get('id'))
        ))
    parents = related_qids('P22', 'P25')
    spouses = related_qids('P26')
    offspring = related_qids('P40')
    predecessor = None
    for _, claim in _claim_values(claims, 'P39'):
        for qualifier in claim.get('qualifiers', {}).get('P1365', []):
            value = qualifier.get('datavalue', {}).get('value', {})
            if isinstance(value, dict) and is_qid(value.get('id')):
                predecessor = value['id']
                break
        if predecessor:
            break
    if predecessor is None:
        predecessors = related_qids('P1365')
        predecessor = predecessors[0] if predecessors else None

    # ── Esclusione pagine non-persona ────
    # Se P31 include tipi come royal house, dynasty, country, skip
    UNWANTED_P31 = {
        "Q4167410",  # Wikimedia disambiguation page
        "Q4167836",  # Wikimedia category
        "Q13417114", # Wikimedia list article
        "Q7278",     # political party
        "Q166376",   # noble house / royal house
        "Q9174",     # country
        "Q6256",     # country
    }
    if any(isinstance(v, dict) and v.get("id") in UNWANTED_P31 for v in p31_values):
        exclusion_reasons[qid] = 'unwanted_P31'
        return None, None, None, None, None

    # Finland's 1918 king-elect never reigned
    if canonical_name in ("Prince Frederick Charles of Hesse",) or wikipedia_title == "Prince Frederick Charles of Hesse":
        has_reign = False
        reign_start = None

    node = (qid, {
        "name": canonical_name,
        "label": canonical_name,
        "wikipedia_title": wikipedia_title,
        "monarch": has_reign,
        "house": house,
        "reign_start": reign_start,
        "birth": birth,
        "death": death,
        #"birth_place": birth_place,
        "birth_country": birth_country,
        "religion": religion,
        "parents": parents,
        "sex": sex,
        "wikidata_qid": qid,
    })

    return node, predecessor, spouses, offspring, parents


Now we can get the list of initial monarchs, from which we will start the search, as well as initialize the list of nodes and edges as well as the search lists.

In [ ]:
# LOAD_SAVED_GRAPH è deciso solo dalla cella di caricamento della cache QID.


In [ ]:
cutoff_year = 1900
nodes = list(G.nodes(data=True)) if LOAD_SAVED_GRAPH else []
edges = list(G.edges(data=True)) if LOAD_SAVED_GRAPH else []
monarchs = []
connections = []
visited_people = set()
if not LOAD_SAVED_GRAPH:
    for title, country in zip(df['monarch_name'], df['country']):
        qid = get_wikidata_qid(title)
        seed_resolution.append({'title': title, 'qid': qid, 'country': country})
        if qid:
            monarchs.append((qid, country))
    unresolved = [row['title'] for row in seed_resolution if row['qid'] is None]
    if unresolved:
        raise RuntimeError(f'Seed senza QID: {unresolved}. Verificare i titoli; non si usano omonimi per tentativi.')


We can now finally explore the connections of our initial monarchs using a BFS algorithm.

The exploration of nodes related via succession is done separately to that of their family connections. This was initially done to explore monarchs in a line of succession differently, and give more importance to the Country of ruling of each. But when growing the graph via more families and older figures, this becomes harder as countries change and so do successions. In this version, all nodes are explored in the same way.

In [ ]:
if not LOAD_SAVED_GRAPH:
    nodes_by_qid = {}

    def collect_person(qid, follow_predecessor=False):
        qid = canonical_qid(qid)
        if qid in visited_people:
            return None
        node, predecessor, spouses, offspring, parents = get_node(qid)
        visited_people.add(qid)
        if node is None:
            return None
        actual_qid, attrs = node
        if actual_qid != qid and actual_qid in visited_people:
            return None
        visited_people.add(actual_qid)
        if attrs['death'] is not None and int(attrs['death']) < cutoff_year:
            exclusion_reasons[actual_qid] = 'death_before_cutoff'
            return None
        nodes_by_qid[actual_qid] = attrs
        if follow_predecessor and predecessor:
            edges.append((actual_qid, predecessor, {'relationship': 'throne'}))
        for relative, relation in [(q, 'marriage') for q in spouses] + [
            (q, 'blood') for q in offspring + parents
        ]:
            edges.append((actual_qid, relative, {'relationship': relation}))
            if canonical_qid(relative) not in visited_people:
                connections.append(relative)
        return predecessor if follow_predecessor else None

    prefetch_entities([q for q, _ in monarchs])
    while monarchs:
        qid, country = monarchs.pop()
        predecessor = collect_person(qid, follow_predecessor=True)
        if predecessor:
            monarchs.append((predecessor, country))
    last_progress = time.monotonic()
    try:
        while connections:
            # L'ordine LIFO e le regole di raccolta restano identici.
            if canonical_qid(connections[-1]) not in entity_cache:
                batch = list(dict.fromkeys(reversed(connections)))[:BATCH_SIZE]
                prefetch_entities(batch)
                prefetch_metadata([q for q in batch if (entity_cache.get(q) or {}).get('sitelinks', {}).get('enwiki', {}).get('title')])
            collect_person(connections.pop())
            if time.monotonic() - last_progress >= 15:
                print(f'Accettati: {len(nodes_by_qid)} | visitati: {len(visited_people)} | '
                      f'in coda: {len(connections)} | richieste HTTP: {HTTP_STATS["requests"]} | '
                      f'recuperi da disco: {HTTP_STATS["disk_hits"]} | '
                      f'minuti: {(time.monotonic()-collection_started)/60:.1f}', flush=True)
                last_progress = time.monotonic()
    finally:
        # Anche le entità già in RAM prima dell'ottimizzazione vengono conservate.
        _disk_store(entity_cache.keys())
    nodes = list(nodes_by_qid.items())
    if not nodes:
        raise RuntimeError('Nessuna persona accettata: il grafo vuoto non sarà salvato.')
    print(f'Raccolti {len(nodes)} QID distinti; {len(edges)} record di relazione.')
    

In [ ]:
def merge_relationships(*relationships):
    """Combine edge relationship labels deterministically in a simple nx.Graph."""
    known_order = ("blood", "marriage", "throne")
    labels = set()
    for relationship in relationships:
        if relationship:
            labels.update(part for part in str(relationship).split("_&_") if part)
    return "_&_".join(
        [label for label in known_order if label in labels]
        + sorted(labels.difference(known_order))
    )


def build_relationship_graph(node_records, edge_records):
    """Preserve all accepted nodes and original edges, merging overlapping labels."""
    graph = nx.Graph(identity_schema='wikidata_qid_v1')
    for qid, attrs in node_records:
        if not is_qid(qid) or attrs.get('wikidata_qid') != qid:
            raise ValueError(f'Nodo con identità incoerente: {qid!r}')
        if any(not is_qid(p) for p in attrs.get('parents', [])):
            raise ValueError(f'Genitore non QID: {qid}')
    graph.add_nodes_from(
        (name, {key: value for key, value in attrs.items() if key != "country"})
        for name, attrs in node_records
    )
    accepted_qids = set(graph.nodes)
    skipped = 0

    for a, b, attrs in edge_records:
        if not is_qid(a) or not is_qid(b):
            raise ValueError(f'Estremi non QID: {a!r}, {b!r}')
        a, b = canonical_qid(a), canonical_qid(b)
        # Manteniamo solo gli estremi effettivamente ammessi dai filtri.
        if a not in accepted_qids or b not in accepted_qids:
            skipped += 1
            discarded_edges.append({
                'source': a, 'target': b, 'relationship': attrs.get('relationship'),
                'missing': {q: exclusion_reasons.get(q, 'not_collected')
                            for q in (a, b) if q not in accepted_qids}
            })
            continue
        incoming = dict(attrs)
        previous = graph[a][b].get("relationship") if graph.has_edge(a, b) else None
        incoming["relationship"] = merge_relationships(previous, incoming.get("relationship"))
        graph.add_edge(a, b, **incoming)

    # If loading the old saved Graph, overwritten blood labels cannot be recovered
    # from its edges. Restore only demonstrable blood+throne overlap from the
    # child's parent attribute, WITHOUT adding edges or nodes.
    for child, attrs in graph.nodes(data=True):
        parents = list(dict.fromkeys(canonical_qid(p) for p in (attrs.get('parents') or [])))
        attrs['parents'] = parents
        if isinstance(parents, (list, tuple, set)):
            for parent in parents:
                if parent in graph and graph.has_edge(child, parent):
                    current = graph[child][parent].get("relationship", "")
                    if "throne" in current.split("_&_"):
                        graph[child][parent]["relationship"] = merge_relationships(current, "blood")

    assert set(graph.nodes) == accepted_qids, "The graph builder changed the accepted node set"
    print(f"Graph construction: {len(accepted_qids)} accepted nodes; {skipped} discarded edge records to excluded QIDs")
    return graph

if not LOAD_SAVED_GRAPH:
    G = build_relationship_graph(nodes, edges)


The original notebook excludes people who died before the 1450 cutoff and removes links to excluded or unparsed pages. The graph builder below preserves exactly the accepted node set.

In [ ]:
# Non mescolare cache precedenti basate sui nomi con il grafo QID.
nodes = list(G.nodes(data=True))
edges = list(G.edges(data=True))
assert all(is_qid(q) and a['wikidata_qid'] == q for q, a in nodes)
assert all(is_qid(p) for _, a in nodes for p in a['parents'])
assert all(a in G and b in G for a, b, _ in edges)
print(f'Grafo QID: {len(G)} persone, {G.number_of_edges()} archi')

G.graph['collection_policy'] = 'human_with_enwiki_death_cutoff_v1'
G.graph['cutoff_year'] = cutoff_year
G.graph['seed_titles_json'] = json.dumps(df['monarch_name'].tolist(), ensure_ascii=False)


Save the corrected graph to a NEW filename, preserving the original graph file.

In [ ]:
if G.number_of_nodes() == 0:
    raise RuntimeError("Refusing to save an empty graph cache")
atomic_pickle_dump(G, GRAPH_FILE)
print(f"Saved corrected graph with {G.number_of_nodes()} nodes to {GRAPH_FILE}")


In [ ]:
import json
from datetime import datetime, timezone
atomic_pickle_dump(nodes, DATA_DIR / f'nodes_{cutoff_year}_qid.pkl')
atomic_pickle_dump(edges, DATA_DIR / f'edges_{cutoff_year}_qid.pkl')
if not LOAD_SAVED_GRAPH:
    audit = {
        'identity_schema': 'wikidata_qid_v1',
        'saved_at_utc': datetime.now(timezone.utc).isoformat(),
        'seeds': seed_resolution, 'exclusions': exclusion_reasons,
        'discarded_edges': discarded_edges, 'qid_aliases': qid_aliases,
        'api_errors': api_errors,
        'note': 'Risposte di una raccolta live; non snapshot storico a una data unica.'
    }
    (DATA_DIR / 'qid_audit.json').write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding='utf-8')
    atomic_pickle_dump(entity_cache, DATA_DIR / 'wikidata_entities_used.pkl')
    atomic_pickle_dump(qid_cache, DATA_DIR / 'seed_title_to_qid.pkl')
print(f'Output QID salvati in {DATA_DIR}')


# TEXT ANALYSIS

1.0) Corpus creation

1)Wikitext exploration.

In this section a firstly a deep analysis of the all the wiki text will be performed to to extract a broad information about the overall word corpus:



* corpus and dictionary word  lenght
* lexical diversity among the corpus
* most common used words



## Wiki text exploration

 This sub-section collects the wikitext of all nodes into a corpus. After tokenization, all words are converted to lowercase and all punctuation is removed. Finally, a customized list of stopwords (nltk english stopwords + frequent wikipedia formatting tokens) is removed for a clean dataset. This process prepares the corpus for efficient natural language processing and analysis within the network graph.

In [ ]:
#Folder  where are saved the txt files with the text of the monarchs
preprocess_dir = str(TEXT_DIR)  # local project data folder
TEXT_DIR.mkdir(parents=True, exist_ok=True)

corpus_root = preprocess_dir
# Solo per l'analisi testuale facoltativa seguente.
for resource in ('punkt', 'punkt_tab', 'stopwords', 'vader_lexicon'):
    nltk.download(resource, download_dir=str(NLTK_DATA_DIR), quiet=True)


In [ ]:
# Stop word list creation:
# Extraction of downloaded nltk english stopwords
stop_words = set(stopwords.words('english'))
# addition of customized stopwords
list_remove =["ref","url", "p", "http","https","www", "org", "com", "publisher", "file", "page", "html", "jpg", "br", "sfn"]
stop_words.update(list_remove)

# Function to calculate vocabulary size
def vocab_size(text):
    """ text: nltk.Text object"""

    words = text.tokens
    unique_words = set(words)
    return len(unique_words)

# function to remove punctuation
def remove_punctuation(tokens):
    # Define a string containing all punctuation characters
    punctuation_chars = string.punctuation +"'' ``"

    # Remove all punctuation characters from the list of tokens
    tokens_without_punctuation = [token  for token in tokens if token not in punctuation_chars]
    return tokens_without_punctuation

# set lower case
def set_lower_case(tokens):
    tokens_lowercase = [token.lower() for token in tokens]
    return tokens_lowercase

##remove punctuation and set lower case functions ##############################

def remove_punctuation_lower(tokens):
    # Define a string containing all punctuation characters
    punctuation_chars = string.punctuation +"'' ``"

    # Remove all punctuation characters from the list of tokens
    tokens_without_punctuation = [token.lower()  for token in tokens if token not in punctuation_chars]
    return tokens_without_punctuation

# function used to calculate the occurency of words in a list and set in a dictionary
def occurencies(list):
    occurencies = {}
    for i in list:
        occurencies[i] = occurencies.get(i, 0) + 1
    return occurencies

In [ ]:
from pathlib import Path

corpus_root = str(Path(corpus_root).resolve())
if corpus_root not in [str(Path(p).resolve()) for p in nltk.data.path]:
    nltk.data.path.append(corpus_root)

corpus = PlaintextCorpusReader(corpus_root, r".*\.txt")

In [ ]:
# corpus creation
corpus = PlaintextCorpusReader(corpus_root, r".*\.txt")
all_files_list = corpus.fileids() # list of alle the files in the corpus ("monarch corpus")
corpus_raw = corpus.raw() # raw text of the entire corpus ("monarch corpus")

# TOKENIZATION of the corpus

# takes to long to run (more then 30 minutes and still running!!)
#tokens = nltk.word_tokenize(corpus_raw) # tokenized corpus = list of tokens
wordlist = corpus.words()  # list of all the words/tokens in the corpus (

tokens_clean_lower = remove_punctuation_lower(wordlist)  # remove punctuation and set lower case
# create a text object from the corpus
text_obj = nltk.Text(tokens_clean_lower)

# vocabulary size (number of unique tokens in the corpus)
tokens_vocabulary_len = vocab_size(text_obj)


##### stop words- removal
tokens_clean_lower_NoStopWords = [word for word in tokens_clean_lower if word.isalpha() and word not in stop_words]
text_obj_NoStopWords = nltk.Text(tokens_clean_lower_NoStopWords)

1) Wikitext exploration:

To get a general overview of the analyzed corpus:

the corpus lenght (number of tokens in the corpus),  the number of unique tokens and the lexical diversity is extract*.

*The lexical diveristy is defined as :

$lexical\_diversity = distinct\_tokens / N\_tokens $ .

In [ ]:
print( "The lenght of the tokens in the corpus  before cleaning is :", len(wordlist) )
print("\n The Vocabulary of all the cleaned corpus has size:", tokens_vocabulary_len )

# lexical diversity among all the wikipedia pages
def lexical_diversity(text):
 if len(text) == 0:
  return 0
 return len(set(text)) / len(text)  # different vocabulary used/all words used

# calculate the percentage
def percentage(count, total):
    return 100 * count / total #percentage

div= lexical_diversity(text_obj)
print("\n Lexical diversity:", div,  f"\n It means that only {round(div*100,3)} % of the words are unique, the other {100- round(div*100,3)} % of the text is composed of the repetition of these words. ")